# Build the Fe(111) + N2 starting configuration

This notebook prepares the initial structure used in the OPES exercise. ASE is
used to construct the Fe(111) slab and place N2 on the surface, while a
pretrained MACE model provides energies and forces for the
geometry optimizations.


### Imports and geometry parameters

The distances below define the initial N2 bond length, the starting N-surface
separation, and the height below which Fe atoms are kept fixed during
relaxation.


In [ ]:
from ase import Atoms
from ase.build import add_adsorbate, bcc111
from ase.constraints import FixAtoms
from ase.io import write
from ase.optimize import QuasiNewton
from ase.visualize import view

from mace.calculators import mace_mp

d_N2 = 1.24       # Initial N-N bond length in Angstrom.
d_N_Fe = 1.15     # Initial height of the N2 molecule above the surface.
z_Fe_fixed = 2.0  # Keep the lower part of the slab fixed during relaxation.


### Build and relax the clean slab

The slab is periodic in the surface plane and non-periodic along the surface
normal. Fixing the bottom layers mimics the bulk support while allowing the
surface atoms to relax.


In [ ]:
slab = bcc111("Fe", size=(3, 4, 8), orthogonal=True)
slab.set_pbc((True, True, False))
slab.set_constraint(FixAtoms(mask=[atom.z < z_Fe_fixed for atom in slab]))

calc = mace_mp(model="mh-0", head="oc20_usemppbe", default_dtype="float64")
slab.calc = calc

slab_relax = QuasiNewton(slab)
slab_relax.run(fmax=0.1)

### Add molecular nitrogen

The N2 molecule is placed approximately parallel to the surface. The subsequent
optimization lets the adsorbate and the top Fe layers adjust from this initial
configuration.


In [ ]:
molecule = Atoms("2N", positions=[(0, 0, 0), (d_N2, 0, 0)])
add_adsorbate(slab, molecule, d_N_Fe, position=(-d_N2 / 2.0, 0), offset=(1.5, 1))

### Relax the adsorbate/surface system

The optimized structure is saved to `opt.traj` so that the relaxation path can
be inspected if needed.


In [ ]:
adsorbate_relax = QuasiNewton(slab, trajectory="opt.traj")
adsorbate_relax.run(fmax=0.1)


### Add vacuum and write the starting configuration

The z cell is enlarged after relaxation to leave vacuum above the slab. The
resulting `init_config.xyz` file is the input used by the OPES simulations.


In [ ]:
cell = slab.get_cell()
cell[2, 2] = slab.get_positions()[:, 2].max() + 20.0
slab.set_cell(cell)

write("init_config.xyz", slab)

### Visualize the final structure

The viewer cell is optional, but useful for checking the slab orientation, fixed
layers, and N2 placement before starting MD simulations.


In [ ]:
view(slab, viewer="x3d")